# BAB IV — HASIL DAN PEMBAHASAN: Klasifikasi Software Inventory GLPI

**Metodologi:** CRISP-DM (6 tahap) | **Input:** `Data_Mentah.csv` / `.xlsx` | **Output:** seluruh berkas di folder **`Hasil/`**

**Perbandingan wajib:** (1) TF-IDF + Logistic Regression, (2) TF-IDF + Linear SVM, (3) TF-IDF + Multinomial Naive Bayes.

> **PENTING TENTANG VALIDITAS:** Label otomatis berbasis kata kunci di notebook ini adalah **label awal demonstrasi**, bukan label resmi GLPI dan bukan *ground truth*. Tanpa file verifikasi ahli, angka evaluasi yang dibuat notebook **HANYA kesesuaian model dengan label otomatis** (proxy), **tidak boleh dinarasikan sebagai akurasi klasifikasi sebenarnya**. Untuk hasil penelitian final, isi `Label_Terverifikasi.xlsx` bersama Pak Widi, kemudian jalankan ulang notebook. Seluruh mode/berkas hasil ditandai jelas.

Dataset asli tidak dinormalisasi di tempat, tidak dihapus, dan tidak dimodifikasi. Semua transformasi dibuat di salinan kerja.

### Peta BAB IV dan nomor coding
| BAB IV | Tahap CRISP-DM | Nomor kode | Output utama |
|---|---|---|---|
| 4.1 | Business Understanding | 01–02 | Diagram CRISP-DM, tujuan/kriteria |
| 4.2 | Data Understanding | 03–07 | Tabel statistik, missing, duplikasi, grafik |
| 4.3 | Data Preparation | 08–13 | `name_clean`, label awal, template validasi, dataset siap model |
| 4.4 | Modeling | 14–16 | Split tanpa kebocoran nama sama, TF-IDF, tiga model |
| 4.5 | Evaluation | 17–19 | Metrik, classification report, confusion matrix |
| 4.6 | Deployment | 20–23 | Prediksi seluruh GLPI, model tersimpan, ringkasan, ZIP |

**Cara menjalankan:** `Runtime → Run all` di Google Colab. Jika file belum ada, Colab akan meminta upload. File `Data_Mentah.csv` harus berasal dari ekspor GLPI mentah (bukan versi simulasi yang diubah namanya).

## 4.1 Business Understanding
### Coding 01 — Import pustaka, konfigurasi, dan folder `Hasil/`

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime
import hashlib, json, os, re, unicodedata, warnings, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
import joblib

SEED = 42
RNG = np.random.default_rng(SEED)
LABELS = ['EXCLUDE_CANDIDATE', 'INCLUDE']
DIR = Path('Hasil')
FOLDERS = {
    'dokumentasi': DIR / '00_Dokumentasi',
    'business': DIR / '01_Business_Understanding',
    'understanding': DIR / '02_Data_Understanding',
    'preparation': DIR / '03_Data_Preparation',
    'modeling': DIR / '04_Modeling',
    'evaluation': DIR / '05_Evaluation',
    'deployment': DIR / '06_Deployment',
    'bab4': DIR / '07_Ringkasan_BAB_IV',
}
for p in FOLDERS.values(): p.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 180, 'axes.titlesize': 13,
                     'axes.labelsize': 10, 'font.size': 10})

def savefig(filename, folder, fig=None):
    fig = fig or plt.gcf()
    fig.savefig(FOLDERS[folder] / filename, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print('Gambar:', FOLDERS[folder] / filename)

def save_csv(frame, filename, folder):
    p = FOLDERS[folder] / filename
    frame.to_csv(p, index=False, encoding='utf-8-sig')
    return p

print('Folder output:', DIR.resolve())

Folder output: C:\Users\magang.tiga\Documents\CODE_GLPI\Hasil


### Coding 02 — Tujuan bisnis dan diagram enam tahap CRISP-DM

**Masalah:** data inventaris GLPI mengandung aplikasi dan komponen pendukung; tim IT memerlukan daftar kandidat INCLUDE/EXCLUDE_CANDIDATE untuk tinjauan. **Output penelitian:** hasil perbandingan ketiga metode dan prediksi yang masih memerlukan keputusan IT.

In [ ]:
objectives = pd.DataFrame([
    ['Tujuan bisnis', 'Membantu memilah software utama dan komponen pendukung untuk kebutuhan peninjauan inventaris IT'],
    ['Target klasifikasi', 'INCLUDE vs EXCLUDE_CANDIDATE'],
    ['Sumber data', 'Ekspor Software Inventory GLPI'],
    ['Metrik evaluasi', 'Accuracy, Precision, Recall, F1-Score, confusion matrix'],
    ['Batas pengambilan keputusan', 'EXCLUDE_CANDIDATE bukan instruksi menghapus software dari GLPI'],
    ['Prasyarat hasil final', 'Label data uji ditetapkan/ditinjau oleh validator IT secara independen'],
], columns=['aspek','penjelasan'])
save_csv(objectives, 'Tabel_4_1_Tujuan_Bisnis.csv', 'business')
fig, ax=plt.subplots(figsize=(13.5,2.6))
stages=['Business\nUnderstanding','Data\nUnderstanding','Data\nPreparation','Modeling','Evaluation','Deployment']
ax.set_xlim(-0.6,5.6); ax.set_ylim(-0.5,1.0); ax.axis('off')
for i,stage in enumerate(stages):
    ax.text(i,0.4,stage,ha='center',va='center',fontweight='bold',fontsize=10,
            bbox=dict(boxstyle='round,pad=0.55',facecolor='#eaf2fa',edgecolor='#57799a'))
    if i<5: ax.annotate('',xy=(i+0.66,0.4),xytext=(i+0.35,0.4),arrowprops=dict(arrowstyle='->',lw=1.7))
ax.set_title('Gambar 4.1 — Alur CRISP-DM Penelitian GLPI',pad=20)
savefig('Gambar_4_1_Alur_CRISP_DM.png','business',fig)
display(objectives)

## 4.2 Data Understanding
### Coding 03 — Membaca file `Data_Mentah` (semua kolom dianggap string agar ID/teks asli tetap utuh)

In [5]:

import pandas as pd
from pathlib import Path

# Lokasi file berdasarkan folder kerja VS Code
input_path = Path("Data_Mentah.csv")

# Membaca CSV menggunakan encoding yang sesuai
for enc in ["utf-8-sig", "cp1252", "latin-1"]:
    try:
        Data_Mentah = pd.read_csv(
            input_path,
            encoding=enc,
            dtype=str,
            keep_default_na=False
        )
        print(f"Berhasil menggunakan encoding: {enc}")
        break
    except UnicodeDecodeError:
        continue

print("Jumlah baris:", Data_Mentah.shape[0])
print("Jumlah kolom:", Data_Mentah.shape[1])

display(Data_Mentah.head())


ParserError: Error tokenizing data. C error: Expected 3 fields in line 16, saw 4


### Coding 04 — Statistik dasar, nilai kosong, duplikasi (tanpa menghapus data)

In [ ]:
raw_name=Data_Mentah['name'].astype(str)
non_empty=raw_name.str.strip().ne('')
summary = pd.DataFrame([
    ['Jumlah record mentah', len(Data_Mentah)],
    ['Jumlah atribut mentah', Data_Mentah.shape[1]],
    ['Jumlah ID berbeda', Data_Mentah['id'].nunique()],
    ['ID duplikat', int(Data_Mentah['id'].duplicated().sum())],
    ['Nama kosong', int((~non_empty).sum())],
    ['Nama unik, termasuk nilai kosong', raw_name.nunique()],
    ['Nama berulang (eksak, baris)', int(raw_name.duplicated().sum())],
    ['Baris duplikat identik semua kolom', int(Data_Mentah.duplicated().sum())],
],columns=['indikator','nilai'])
missing = pd.DataFrame({'kolom':Data_Mentah.columns,
  'jumlah_kosong':[int(Data_Mentah[c].astype(str).str.strip().eq('').sum()) for c in Data_Mentah.columns]})
missing['persentase']= (100*missing['jumlah_kosong']/len(Data_Mentah)).round(2)
missing=missing.sort_values('jumlah_kosong',ascending=False)
save_csv(summary,'Tabel_4_2_Ringkasan_Data_Mentah.csv','understanding')
save_csv(missing,'Tabel_4_3_Nilai_Kosong.csv','understanding')
print('Ringkasan:'); display(summary)
print('Kolom kosong terbanyak:'); display(missing.head(12))

### Coding 05 — Visualisasi nilai kosong dan kategori software

In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
sub=missing.head(12).iloc[::-1]
ax.barh(sub['kolom'],sub['jumlah_kosong'])
ax.set(title='Gambar 4.2 — Jumlah Nilai Kosong per Atribut',xlabel='Jumlah baris kosong')
ax.grid(axis='x',alpha=.2);savefig('Gambar_4_2_Nilai_Kosong.png','understanding',fig)

col='software_category_name'
if col in Data_Mentah:
    cat=Data_Mentah[col].replace('', '(Kosong)').value_counts().rename_axis('kategori').reset_index(name='jumlah')
    save_csv(cat,'Tabel_4_4_Distribusi_Kategori.csv','understanding')
    fig,ax=plt.subplots(figsize=(10,5.8))
    t=cat.head(12).iloc[::-1]
    ax.barh(t['kategori'],t['jumlah'])
    ax.set(title='Gambar 4.3 — Kategori Software GLPI (12 Teratas)',xlabel='Jumlah record')
    ax.grid(axis='x',alpha=.2);savefig('Gambar_4_3_Distribusi_Kategori.png','understanding',fig)
    display(cat.head(12))

### Coding 06 — Grafik produsen software dan duplikasi nama

In [ ]:
if 'manufacturer_name' in Data_Mentah:
    mf=(Data_Mentah['manufacturer_name'].replace('', '(Kosong)').value_counts()
        .rename_axis('manufacturer').reset_index(name='jumlah'))
    save_csv(mf,'Tabel_4_5_Distribusi_Manufacturer.csv','understanding')
    fig,ax=plt.subplots(figsize=(11,6))
    q=mf.head(12).iloc[::-1]
    ax.barh(q['manufacturer'].str.slice(0,46),q['jumlah'])
    ax.set(title='Gambar 4.4 — 12 Manufacturer dengan Record Terbanyak',xlabel='Jumlah record')
    ax.grid(axis='x',alpha=.2);savefig('Gambar_4_4_Distribusi_Manufacturer.png','understanding',fig)
name_freq=(raw_name[non_empty].value_counts().rename_axis('name').reset_index(name='jumlah_record'))
save_csv(name_freq,'Tabel_4_6_Frekuensi_Nama_Software.csv','understanding')
fig,ax=plt.subplots(figsize=(11,5.5))
t=name_freq.head(12).iloc[::-1]
ax.barh(t['name'].str.slice(0,55),t['jumlah_record'])
ax.set(title='Gambar 4.5 — Nama Software yang Paling Sering Muncul',xlabel='Jumlah record')
ax.grid(axis='x',alpha=.2);savefig('Gambar_4_5_Duplikasi_Nama.png','understanding',fig)

### Coding 07 — Distribusi panjang nama sebelum preprocessing

In [ ]:
raw_len=raw_name.str.len()
fig,ax=plt.subplots(figsize=(9,4.5))
ax.hist(raw_len,bins=35)
ax.set(title='Gambar 4.6 — Distribusi Panjang Nama Software Mentah',xlabel='Jumlah karakter',ylabel='Jumlah record')
ax.grid(axis='y',alpha=.2);savefig('Gambar_4_6_Panjang_Nama_Mentah.png','understanding',fig)

## 4.3 Data Preparation
### Coding 08 — Cleaning dan case folding menjadi `name_clean`

Tahapan ini **tidak mengubah kolom `name`**. Hanya memangkas spasi, menyamakan huruf kecil dan bentuk Unicode, serta karakter kontrol. Simbol versi/titik/`+`/`-` yang berpotensi penting tidak dihapus secara agresif.

In [ ]:
def clean_name(value):
    s=unicodedata.normalize('NFKC',str(value or ''))
    s=s.casefold()
    s=re.sub(r'[\t\n\r\f\v]+',' ',s)
    s=re.sub(r'\s+',' ',s).strip()
    return s
Data_Proses = Data_Mentah.copy()
Data_Proses['name_clean']=Data_Proses['name'].map(clean_name)
Data_Proses['teks_model']=Data_Proses['name_clean']  # Hanya nama software, bukan atribut label/kategori
Data_Proses['nama_berubah']=Data_Proses['name'].ne(Data_Proses['name_clean'])
Data_Proses['nama_duplikat_berdasarkan_clean']=Data_Proses['name_clean'].duplicated(keep=False)
Data_Proses['jumlah_karakter_sebelum']=Data_Proses['name'].str.len()
Data_Proses['jumlah_karakter_sesudah']=Data_Proses['name_clean'].str.len()
prep_stats=pd.DataFrame([
 ['Record masuk',len(Data_Mentah)],['Nama kosong sebelum',int(Data_Mentah.name.str.strip().eq('').sum())],
 ['Nama kosong sesudah',int(Data_Proses.name_clean.eq('').sum())],
 ['Nama yang mengalami perubahan penulisan',int(Data_Proses.nama_berubah.sum())],
 ['Nama unik sebelum (termasuk kosong)',Data_Mentah.name.nunique()],
 ['Nama unik setelah cleaning (termasuk kosong)',Data_Proses.name_clean.nunique()],
 ['Record tetap dipertahankan',len(Data_Proses)]],columns=['indikator','jumlah'])
save_csv(prep_stats,'Tabel_4_7_Statistik_Preprocessing.csv','preparation')
examples=Data_Proses[['id','name','name_clean']].loc[Data_Proses['nama_berubah']].head(40).copy()
save_csv(examples,'Tabel_4_8_Contoh_Sebelum_Sesudah.csv','preparation')
fig,ax=plt.subplots(figsize=(8,4.5))
ax.bar(['Tidak berubah','Berubah'],[int((~Data_Proses.nama_berubah).sum()),int(Data_Proses.nama_berubah.sum())])
ax.set(title='Gambar 4.7 — Pengaruh Preprocessing Nama',ylabel='Jumlah record')
savefig('Gambar_4_7_Dampak_Preprocessing.png','preparation',fig)
display(prep_stats);display(examples.head(10))

### Coding 09 — Memeriksa frekuensi kata untuk dokumentasi text preprocessing

In [ ]:
def get_tokens(s):
    return re.findall(r'(?u)[a-z0-9][a-z0-9._+\-]*',s)
tokens=Counter(t for txt in Data_Proses['name_clean'] for t in get_tokens(txt) if len(t)>1)
terms=pd.DataFrame(tokens.most_common(100),columns=['token','frekuensi'])
save_csv(terms,'Tabel_4_9_Top_Token.csv','preparation')
fig,ax=plt.subplots(figsize=(10,5.3));t=terms.head(20).iloc[::-1]
ax.barh(t.token,t.frekuensi);ax.set(title='Gambar 4.8 — 20 Token Software Terbanyak',xlabel='Frekuensi kemunculan')
savefig('Gambar_4_8_Top_Token.png','preparation',fig)

### Coding 10 — Automatic labeling **awal, hanya eksperimen berbasis kata kunci**

**Penting:** Kode ini **tidak** mereproduksi 67 aturan GLPI dan **tidak** menyatakan semua data sudah berlabel. Label sementara diisi hanya untuk pola yang cukup eksplisit; software lain menjadi `REVIEW` (label kosong). `EXCLUDE_CANDIDATE` artinya kandidat tidak ditampilkan di daftar software utama, **bukan** perintah menghapus instalasi.

In [ ]:
# Urutan penting: pengecualian yang eksplisit diperiksa terlebih dahulu
EXCLUDE_REGEX = [
 (r'\blanguage\s*pack\b|\b(?:language|localization|localisation)\s*(?:resources?|files?)\b','Paket bahasa/lokalisasi'),
 (r'\bredistributable\b|\bvisual\s*c\+\+\s*(?:20\d\d\s*)?(?:redistributable|runtime)\b','Redistributable/runtime'),
 (r'\b(?:desktop|windows|webview2|java|dotnet|\.net)\s*runtime\b|\bwindows\s*app\s*runtime\b','Runtime pendukung'),
 (r'\b(?:cumulative|security|quality)\s*update\b|\bhotfix\b|\bkb\s*\d{6,8}\b','Hotfix/update'),
 (r'\b(?:printer|display|graphics|audio|network|bluetooth|chipset|usb|touchpad|camera)\s*drivers?\b','Driver perangkat'),
 (r'\b(?:add[ -]?ins?|plugins?|plug[ -]?ins?)\b','Add-in/plugin'),
 (r'\b(?:help|documentation|tutorial|sample)\s*files?\b','File dokumentasi/sampel'),
]
INCLUDE_REGEX = [
 (r'\b(?:microsoft|ms)\s*(?:office|visio|project|word|excel|powerpoint)\b|\boffice\s*(?:201\d|202\d|365)\b','Aplikasi produktivitas'),
 (r'\b(?:google\s*chrome|mozilla\s*firefox|microsoft\s*edge|opera\s*browser)\b','Browser'),
 (r'\b(?:adobe\s*(?:acrobat|photoshop|illustrator|premiere|indesign|lightroom))\b','Aplikasi kreatif/PDF'),
 (r'\b(?:autocad|autodesk\s*revit|civil\s*3d|solidworks|arcgis|qgis)\b','Aplikasi CAD/GIS'),
 (r'\b(?:microsoft\s*teams|zoom\s*(?:workplace|meetings|client)|slack\s*desktop)\b','Aplikasi komunikasi'),
 (r'\b(?:7[ -]?zip|winrar|notepad\+\+|visual\s*studio\s*code|dbeaver|gimp)\b','Aplikasi utilitas/pengembangan'),
]
EXCLUDE_PATTERNS=[(re.compile(rx,re.I),reason) for rx,reason in EXCLUDE_REGEX]
INCLUDE_PATTERNS=[(re.compile(rx,re.I),reason) for rx,reason in INCLUDE_REGEX]

def auto_label(name):
    if not name: return '', 'REVIEW', 'Nama software kosong'
    for pat,reason in EXCLUDE_PATTERNS:
        if pat.search(name):return 'EXCLUDE_CANDIDATE','HEURISTIC_UNVERIFIED',reason
    for pat,reason in INCLUDE_PATTERNS:
        if pat.search(name):return 'INCLUDE','HEURISTIC_UNVERIFIED',reason
    return '', 'REVIEW', 'Pola belum teridentifikasi; perlu tinjauan IT'

lab=pd.DataFrame([auto_label(s) for s in Data_Proses['name_clean']],
                 columns=['label_auto','label_source','label_reason'],index=Data_Proses.index)
Data_Proses=pd.concat([Data_Proses,lab],axis=1)
label_dist=(Data_Proses.label_auto.replace('', 'REVIEW / KOSONG').value_counts()
            .rename_axis('status_label').reset_index(name='jumlah'))
save_csv(label_dist,'Tabel_4_10_Distribusi_Label_Awal.csv','preparation')
save_csv(Data_Proses,'Dataset_Preprocessing_dan_Label_Awal.csv','preparation')
fig,ax=plt.subplots(figsize=(9,4.5));ax.bar(label_dist.status_label,label_dist.jumlah)
ax.set(title='Gambar 4.9 — Distribusi Label Otomatis Sementara',ylabel='Jumlah record')
ax.tick_params(axis='x',rotation=12);savefig('Gambar_4_9_Label_Awal.png','preparation',fig)
display(label_dist)

### Coding 11 — Menyiapkan sampel verifikasi untuk Pak Widi (tanpa membuat klaim sudah divalidasi)

Notebook memilih hingga 2.000 baris dari **12.390 data mentah** dengan campuran label awal dan REVIEW. Kolom `label_verified` **sengaja kosong**. Pengisian harus benar-benar dilakukan oleh validator. Semua ID adalah ID ekspor GLPI, tidak diganti.

In [ ]:
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

REVIEW_N=min(2000,len(Data_Proses))
# Sampling deterministik campuran, tidak memakai file 2.000 lama.
# Prioritaskan variasi nama melalui satu nama bersih per sampel apabila mungkin.
unique_for_review=Data_Proses.sort_values('id').drop_duplicates('name_clean').copy()
unique_for_review['bucket']=unique_for_review.label_auto.replace('', 'REVIEW')
sections=[]
for bucket in ['INCLUDE','EXCLUDE_CANDIDATE','REVIEW']:
    g=unique_for_review[unique_for_review.bucket.eq(bucket)]
    quota=round(REVIEW_N/3)
    sections.append(g.sample(min(len(g),quota),random_state=SEED))
chosen=pd.concat(sections).drop_duplicates('id')
if len(chosen)<REVIEW_N:
    remain=unique_for_review.loc[~unique_for_review.id.isin(chosen.id)]
    chosen=pd.concat([chosen,remain.sample(min(REVIEW_N-len(chosen),len(remain)),random_state=SEED)])
chosen=chosen.head(REVIEW_N).sort_values('id',key=lambda s:pd.to_numeric(s.str.replace(',',''),errors='coerce').fillna(0))
review_cols=['id','name','manufacturer_name','software_category_name','label_auto','label_reason']
for c in review_cols:
    if c not in chosen:chosen[c]=''
template=chosen[review_cols].copy()
template['label_verified']=''  # hanya boleh diisi validator
# Cadangan kolom untuk jejak audit
for c in ['catatan_validator','nama_validator','tanggal_verifikasi']:
    template[c]=''
save_csv(template,'Template_Validasi_Pak_Widi.csv','preparation')
wb=Workbook();ws=wb.active;ws.title='Validasi_2000'
ws.append(list(template.columns))
for row in template.itertuples(index=False,name=None):ws.append(list(row))
ws.freeze_panes='C2'; ws.auto_filter.ref=ws.dimensions
for c in ws[1]:c.fill=PatternFill('solid',fgColor='183B5B');c.font=Font(color='FFFFFF',bold=True)
widths={'A':15,'B':57,'C':31,'D':27,'E':24,'F':40,'G':25,'H':44,'I':25,'J':23}
for col,w in widths.items():ws.column_dimensions[col].width=w
ws.row_dimensions[1].height=28
for r in range(2,ws.max_row+1):
    for c in (7,8,9,10):
        cell=ws.cell(r,c);cell.fill=PatternFill('solid',fgColor='E8F1FE');cell.font=Font(color='154C9C')
dv=DataValidation(type='list',formula1='"INCLUDE,EXCLUDE_CANDIDATE,TIDAK_PASTI"')
ws.add_data_validation(dv);dv.add(f'G2:G{ws.max_row}')
instructions=wb.create_sheet('Petunjuk')
for line in [
 ('PETUNJUK VALIDASI DATA', ''),
 ('1. Jangan ubah id dan name', 'ID dipakai untuk menggabungkan koreksi kembali ke dataset mentah.'),
 ('2. Isi label_verified', 'INCLUDE / EXCLUDE_CANDIDATE / TIDAK_PASTI.'),
 ('3. label_auto hanya usulan', 'Label awal berasal dari kata kunci eksperimen; bisa salah.'),
 ('4. Isi nama dan tanggal', 'Agar sumber verifikasi dapat dipertanggungjawabkan.'),
 ('5. Simpan file', 'Gunakan nama Label_Terverifikasi.xlsx di folder notebook lalu jalankan ulang.'),
 ('6. Ground truth', 'Hanya nilai berlabel yang benar-benar diperiksa ahli dapat dianggap terverifikasi.'),
]:instructions.append(line)
instructions.column_dimensions['A'].width=34;instructions.column_dimensions['B'].width=105
for cell in instructions[1]:cell.font=Font(bold=True,color='183B5B')
review_file=FOLDERS['preparation']/'Template_Validasi_Pak_Widi.xlsx'
wb.save(review_file)
print('Template dibuat:',review_file,'| total:',len(template),'baris (belum diverifikasi)')

### Coding 12 — Membaca label terverifikasi **jika tersedia**, tanpa mencampuradukkan dengan label otomatis

Opsional: unggah `Label_Terverifikasi.xlsx` (dari template pada kode 11) ke folder kerja notebook, atau `Label_Terverifikasi.csv`. Jika belum ada, notebook tetap berjalan dalam **MODE DEMO / PROXY**. Jika ada, data uji diambil khusus dari data ahli yang dinyatakan terverifikasi.

In [ ]:
review_paths=[Path('Label_Terverifikasi.xlsx'),Path('Label_Terverifikasi.csv')]
verify_path=next((p for p in review_paths if p.exists()),None)
expert=pd.DataFrame(columns=['id','label_verified'])
if verify_path:
    if verify_path.suffix=='.csv':
        expert=pd.read_csv(verify_path,dtype=str,keep_default_na=False,encoding='utf-8-sig')
    else:
        xl=pd.ExcelFile(verify_path)
        sheet='Validasi_2000' if 'Validasi_2000' in xl.sheet_names else xl.sheet_names[0]
        expert=pd.read_excel(verify_path,sheet_name=sheet,dtype=str,keep_default_na=False)
    expert.columns=expert.columns.str.strip()
    if 'label_verified' not in expert and 'label' in expert:expert=expert.rename(columns={'label':'label_verified'})
    assert {'id','label_verified'}.issubset(expert), 'File verifikasi memerlukan kolom id dan label_verified.'
    expert['id']=expert.id.astype(str).str.replace(',','',regex=False).str.strip()
    expert['label_verified']=expert.label_verified.astype(str).str.strip().str.upper()
    valid=expert.label_verified.isin(LABELS)
    invalid=expert.label_verified.ne('') & ~expert.label_verified.isin(LABELS+['TIDAK_PASTI','REVIEW'])
    if invalid.any():
        raise ValueError(f'Ditemukan {invalid.sum()} label tidak dikenal di file verifikasi. Pilih INCLUDE / EXCLUDE_CANDIDATE / TIDAK_PASTI.')
    expert=expert.loc[valid].copy()
    if expert['id'].duplicated().any():
        raise ValueError('Ada ID berulang pada berkas verifikasi. Perbaiki terlebih dahulu.')
    # ID dapat pernah diunduh dengan pemisah ribuan, di sini disamakan hanya saat join
    srcids=Data_Proses['id'].str.replace(',','',regex=False).str.strip()
    unknown=expert.loc[~expert.id.isin(srcids),'id']
    if len(unknown):raise ValueError(f'{len(unknown)} ID verifikasi tidak ditemukan di Data_Mentah.')
    print('File verifikasi terbaca:',verify_path,'| label valid:',len(expert))
else:
    print('BELUM ADA validasi ahli. Seluruh hasil evaluasi akan ditandai DEMO / PROXY.')

Data_Proses['join_id']=Data_Proses['id'].str.replace(',','',regex=False).str.strip()
Data_Proses=Data_Proses.merge(expert[['id','label_verified']].rename(columns={'id':'join_id'}),on='join_id',how='left',validate='many_to_one')
Data_Proses['label_verified']=Data_Proses['label_verified'].fillna('')
Data_Proses['label_effective']=np.where(Data_Proses.label_verified.isin(LABELS),
  Data_Proses.label_verified,Data_Proses.label_auto)
Data_Proses['sumber_label_effective']=np.where(Data_Proses.label_verified.isin(LABELS),
  'EXPERT_VERIFIED', np.where(Data_Proses.label_auto.isin(LABELS),'HEURISTIC_UNVERIFIED','UNLABELED_REVIEW'))

verified_count=int(Data_Proses.label_verified.isin(LABELS).sum())
verified_class=(Data_Proses.loc[Data_Proses.label_verified.isin(LABELS),'label_verified'].value_counts())
MODE_EVALUASI='GROUND_TRUTH_VERIFIED' if (verified_count>=30 and all(verified_class.get(c,0)>=10 for c in LABELS)) else 'DEMO_PROXY_UNVERIFIED'
if verified_count and MODE_EVALUASI.startswith('DEMO'):
    print('PERINGATAN: label ahli kurang untuk evaluasi dua kelas (minimal 10 tiap kelas dan 30 total). Tetap DEMO.')
print('MODE EVALUASI:',MODE_EVALUASI,'| jumlah verified:',verified_count)
save_csv(Data_Proses,'Dataset_Gabungan_Label.csv','preparation')

### Coding 13 — Menyiapkan unit unik untuk pemodelan dan menangani konflik label

**Raw 12.390 baris tidak dihapus.** Namun, contoh dengan `name_clean` sama tidak boleh tersebar di train dan test (*data leakage*). Untuk training, satu nama bersih digunakan sebagai satu unit; seluruh nama yang sama ada di kelompok yang sama. Bila label antar-ID bertentangan dan tidak ada keputusan verifikasi, kelompok tersebut ditandai konflik dan tidak digunakan saat pelatihan.

In [ ]:
usable=Data_Proses.loc[Data_Proses.name_clean.ne('')].copy()
rows=[]; conflicts=[]
for name,g in usable.groupby('name_clean',sort=False):
    verified=set(g.loc[g.label_verified.isin(LABELS),'label_verified'])
    auto=set(g.loc[g.label_auto.isin(LABELS),'label_auto'])
    if len(verified)>1:
        conflicts.append({'name_clean':name,'id_contoh':g.iloc[0]['id'],'jenis_konflik':'Verified berbeda pada nama identik','label_terlihat':'; '.join(sorted(verified))})
        continue
    if verified:
        lab=next(iter(verified));origin='EXPERT_VERIFIED'
    elif len(auto)>1:
        conflicts.append({'name_clean':name,'id_contoh':g.iloc[0]['id'],'jenis_konflik':'Heuristik berbeda pada nama identik','label_terlihat':'; '.join(sorted(auto))})
        continue
    elif auto:
        lab=next(iter(auto));origin='HEURISTIC_UNVERIFIED'
    else:continue
    rows.append({'name_clean':name,'text_model':name,'label':lab,'sumber_label':origin,
                 'jumlah_record_raw':len(g),'id_contoh':g.iloc[0]['id']})
unit_model=pd.DataFrame(rows)
konflik_df=pd.DataFrame(conflicts,columns=['name_clean','id_contoh','jenis_konflik','label_terlihat'])
save_csv(konflik_df,'Tabel_4_11_Konflik_Label_Nama.csv','preparation')
save_csv(unit_model,'Dataset_Unit_Utama_Siap_Model.csv','preparation')
print('Unit nama unik berlabel:',len(unit_model),'| konflik nama:',len(konflik_df))
print('Distribusi unit label:');display(unit_model.groupby(['sumber_label','label']).size().rename('jumlah').reset_index())
assert unit_model['name_clean'].nunique()==len(unit_model)

## 4.4 Modeling
### Coding 14 — Split 80:20 dan perlindungan data leakage

**Mode DEMO:** test memakai sebagian label otomatis untuk menguji apakah model dapat meniru pola aturan (bukan mengukur akurasi di dunia nyata). **Mode GROUND_TRUTH:** test berisi hanya nama-nama berlabel hasil verifikasi ahli, sementara training boleh memuat label heuristik dan hasil verifikasi di luar test. Data test tidak masuk proses fit TF-IDF.

In [ ]:
def stratified_split(frame,test_size=.20,seed=SEED):
    if frame['label'].nunique()!=2 or frame['label'].value_counts().min()<2:
        raise ValueError('Setiap kelas minimal dua contoh nama unik untuk train/test split.')
    return train_test_split(frame, test_size=test_size, random_state=seed, stratify=frame['label'])

verified_units=unit_model.loc[unit_model.sumber_label.eq('EXPERT_VERIFIED')].copy()
if MODE_EVALUASI=='GROUND_TRUTH_VERIFIED':
    cls=verified_units.label.value_counts()
    if (len(verified_units)<30 or any(cls.get(c,0)<10 for c in LABELS)):
        MODE_EVALUASI='DEMO_PROXY_UNVERIFIED'

if MODE_EVALUASI=='GROUND_TRUTH_VERIFIED':
    verified_train, test_units=stratified_split(verified_units)
    train_units=unit_model.loc[~unit_model.name_clean.isin(test_units.name_clean)].copy()
else:
    train_units,test_units=stratified_split(unit_model)

assert set(train_units.name_clean).isdisjoint(set(test_units.name_clean)), 'Data leakage: nama sama di train dan test.'
assert set(LABELS).issubset(set(train_units.label)) and set(LABELS).issubset(set(test_units.label))
X_train=train_units.text_model.astype(str);y_train=train_units.label.astype(str)
X_test=test_units.text_model.astype(str); y_test=test_units.label.astype(str)
train_units=train_units.assign(split='TRAIN');test_units=test_units.assign(split='TEST')
save_csv(train_units,'Data_Train.csv','modeling')
save_csv(test_units,'Data_Test.csv','modeling')
split_report=pd.concat([
 train_units.groupby('label').size().rename('jumlah').reset_index().assign(split='TRAIN'),
 test_units.groupby('label').size().rename('jumlah').reset_index().assign(split='TEST')],ignore_index=True)
save_csv(split_report,'Tabel_4_12_Distribusi_Train_Test.csv','modeling')
fig,ax=plt.subplots(figsize=(8,4.5))
q=split_report.pivot(index='split',columns='label',values='jumlah').reindex(index=['TRAIN','TEST']).fillna(0)
q.plot(kind='bar',stacked=True,ax=ax,rot=0);ax.set(title='Gambar 4.10 — Distribusi Unit Nama Train dan Test',ylabel='Unit nama unik')
savefig('Gambar_4_10_Split_Train_Test.png','modeling',fig)
print('MODE:',MODE_EVALUASI,'| train:',len(train_units),'| test:',len(test_units));display(split_report)

### Coding 15 — TF-IDF + Logistic Regression, Linear SVM, Multinomial Naive Bayes

Ketiga model menggunakan **pengaturan TF-IDF identik**, split identik, dan fitur **hanya `name_clean`**. `Pipeline` memastikan TF-IDF hanya *fit* pada data training.

In [ ]:
TFIDF_PARAMS=dict(lowercase=False,ngram_range=(1,2),min_df=2,max_df=.95,
                  max_features=30000,sublinear_tf=True,
                  token_pattern=r'(?u)[a-z0-9][a-z0-9._+\-]*')
MODEL_SPECS={
 'Logistic_Regression': LogisticRegression(max_iter=1500,random_state=SEED,class_weight='balanced'),
 'Linear_SVM': LinearSVC(random_state=SEED,class_weight='balanced'),
 'Multinomial_Naive_Bayes': MultinomialNB(alpha=1.0)
}
pipelines={}
for name,est in MODEL_SPECS.items():
    pipe=Pipeline([('tfidf',TfidfVectorizer(**TFIDF_PARAMS)),('classifier',est)])
    pipe.fit(X_train,y_train)
    pipelines[name]=pipe
    joblib.dump(pipe,FOLDERS['modeling']/f'Model_{name}.joblib')
    print(name,':',len(pipe.named_steps['tfidf'].get_feature_names_out()),'fitur TF-IDF')

# Contoh bobot TF-IDF yang bisa dimasukkan ke BAB IV
vec=pipelines['Logistic_Regression'].named_steps['tfidf']
features=vec.get_feature_names_out()
example_names=X_train.iloc[:min(5,len(X_train))].tolist()
weighted=vec.transform(example_names)
rows=[]
for idx,text in enumerate(example_names):
    v=weighted.getrow(idx)
    top=sorted(zip(v.indices,v.data),key=lambda z:-z[1])[:8]
    for col,w in top:rows.append({'id_contoh':idx+1,'name_clean':text,'term':features[col],'tfidf_weight':round(float(w),6)})
tfidf_example=pd.DataFrame(rows)
save_csv(tfidf_example,'Tabel_4_13_Contoh_Bobot_TFIDF.csv','modeling')
display(tfidf_example.head(15))

### Coding 16 — Tabel perbandingan konfigurasi tiga model dan validasi silang

Validasi silang dijalankan **hanya pada training set nama unik**. Saat belum ada verifikasi ahli, skornya juga **bersifat DEMO / PROXY**.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
cv_rows=[]
minclass=int(y_train.value_counts().min())
folds=min(5,minclass)
if folds>=2:
    kfold=StratifiedKFold(n_splits=folds,shuffle=True,random_state=SEED)
    for name in MODEL_SPECS:
        scores=cross_val_score(pipelines[name],X_train,y_train,cv=kfold,scoring='f1_macro',n_jobs=1)
        cv_rows.append({'model':name,'jumlah_fold':folds,
                        'f1_macro_cv_mean':float(scores.mean()),
                        'f1_macro_cv_std':float(scores.std()),'mode_evaluasi':MODE_EVALUASI})
cv_df=pd.DataFrame(cv_rows)
save_csv(cv_df,'Tabel_4_14_Validasi_Silang_Train.csv','modeling')
if len(cv_df):display(cv_df)

## 4.5 Evaluation
### Coding 17 — Accuracy, Precision, Recall, F1-Score dan confusion matrix untuk tiga model

**Interpretasi berdasarkan `MODE_EVALUASI`:** `DEMO_PROXY_UNVERIFIED` = hanya kesesuaian dengan *label aturan*. `GROUND_TRUTH_VERIFIED` = pengujian pada sampel dengan label ahli. Bahkan pada mode kedua, jelaskan bahwa sebagian training masih dapat menggunakan label awal heuristik.

In [ ]:
metric_rows=[];prediction_tests={}
for name,pipe in pipelines.items():
    pred=pipe.predict(X_test)
    prediction_tests[name]=pred
    prec,rec,f1,_=precision_recall_fscore_support(y_test,pred,average='macro',zero_division=0)
    p_inc,r_inc,f_inc,_=precision_recall_fscore_support(y_test,pred,labels=['INCLUDE'],average='macro',zero_division=0)
    metric_rows.append({'model':name,'accuracy':accuracy_score(y_test,pred),
                        'precision_macro':prec,'recall_macro':rec,'f1_macro':f1,
                        'f1_include':f_inc,'train_n':len(train_units),'test_n':len(test_units),
                        'mode_evaluasi':MODE_EVALUASI})
    report=pd.DataFrame(classification_report(y_test,pred,labels=LABELS,
                     output_dict=True,zero_division=0)).T.reset_index(names='kelas')
    save_csv(report,f'Classification_Report_{name}.csv','evaluation')
    matrix=confusion_matrix(y_test,pred,labels=LABELS)
    cm=pd.DataFrame(matrix,index=LABELS,columns=LABELS)
    cm.index.name='aktual';cm.reset_index().to_csv(FOLDERS['evaluation']/f'Confusion_Matrix_{name}.csv',index=False,encoding='utf-8-sig')
    fig,ax=plt.subplots(figsize=(7,5))
    disp=ConfusionMatrixDisplay(matrix,display_labels=LABELS)
    disp.plot(ax=ax,values_format='d',colorbar=False,xticks_rotation=18)
    ax.set_title('Gambar 4.11 — Confusion Matrix '+name.replace('_',' '))
    savefig(f'Gambar_4_11_CM_{name}.png','evaluation',fig)

metrics=pd.DataFrame(metric_rows).sort_values('f1_macro',ascending=False).reset_index(drop=True)
save_csv(metrics,'Tabel_4_15_Perbandingan_Kinerja_3_Model.csv','evaluation')
test_predictions=test_units[['id_contoh','name_clean','label']].copy().rename(columns={'label':'label_acuan'})
for name,pred in prediction_tests.items():test_predictions[f'pred_{name}']=pred
save_csv(test_predictions,'Prediksi_Data_Test_3_Model.csv','evaluation')
print('MODE:',MODE_EVALUASI,' (baca sebelum menyimpulkan hasil)')
display(metrics)

### Coding 18 — Grafik perbandingan metrik ketiga algoritma

In [ ]:
fig,ax=plt.subplots(figsize=(11.5,5.5))
cols=['accuracy','precision_macro','recall_macro','f1_macro']
xx=np.arange(len(metrics));w=.19
for j,col in enumerate(cols):ax.bar(xx+(j-1.5)*w,metrics[col],width=w,label=col)
ax.set_xticks(xx,metrics.model.str.replace('_',' ',regex=False),rotation=12,ha='right')
ax.set_ylim(0,1.08);ax.set_ylabel('Nilai metrik (0–1)')
ax.set_title('Gambar 4.12 — Perbandingan Kinerja: '+('DEMO / PROXY' if MODE_EVALUASI.startswith('DEMO') else 'DATA UJI TERVERIFIKASI'))
ax.legend(ncol=4,loc='upper center');ax.grid(axis='y',alpha=.2)
savefig('Gambar_4_12_Perbandingan_3_Model.png','evaluation',fig)

### Coding 19 — Kata/fitur yang paling memengaruhi klasifikasi (interpretasi model)

Bukti bobot model bukan otomatis bukti keputusan yang benar; digunakan sebagai bahan diskusi BAB IV mengenai kata penanda software utama atau komponen.

In [ ]:
feature_rows=[]
for name,pipe in pipelines.items():
    vocab=pipe.named_steps['tfidf'].get_feature_names_out()
    clf=pipe.named_steps['classifier']
    # Urutan dua kelas sklearn disimpan di classes_
    classes=list(clf.classes_)
    pos=classes.index('INCLUDE');neg=classes.index('EXCLUDE_CANDIDATE')
    if hasattr(clf,'coef_'):
        weights=np.ravel(clf.coef_)
        if classes[1]!='INCLUDE':weights=-weights
    else:
        weights=clf.feature_log_prob_[pos]-clf.feature_log_prob_[neg]
    indices=np.argsort(weights)
    for side,selection in [('INCLUDE',indices[-15:][::-1]),('EXCLUDE_CANDIDATE',indices[:15])]:
        for i in selection:
            feature_rows.append({'model':name,'indikasi_label':side,'term':vocab[i],'bobot':float(weights[i])})
    # Grafik bobot positif/negatif paling menonjol
    ix=list(indices[:8])+list(indices[-8:])
    fig,ax=plt.subplots(figsize=(9,6.5))
    ax.barh(np.arange(len(ix)),weights[ix]);ax.set_yticks(np.arange(len(ix)),vocab[ix]);
    ax.axvline(0,color='black',linewidth=.7)
    ax.set(title='Gambar 4.13 — Fitur Penting: '+name.replace('_',' '),xlabel='Bobot untuk arah kelas INCLUDE (+) vs EXCLUDE (−)')
    savefig(f'Gambar_4_13_Fitur_Penting_{name}.png','evaluation',fig)
feature_df=pd.DataFrame(feature_rows)
save_csv(feature_df,'Tabel_4_16_Fitur_Penting_Setia_Model.csv','evaluation')

## 4.6 Deployment
### Coding 20 — Prediksi SELURUH dataset GLPI menggunakan ketiga model

Data mentah dipertahankan. Hasil prediksi tiga model **bukan** keputusan penghapusan software. Majority vote dibuat sebagai referensi, termasuk pada record yang sudah punya label awal. Nama kosong akan dibiarkan tanpa prediksi.

In [ ]:
Data_Prediksi=Data_Proses.copy()
valid_names=Data_Prediksi.teks_model.ne('')
for name,pipe in pipelines.items():
    out_col='pred_'+name
    Data_Prediksi[out_col]=''
    Data_Prediksi.loc[valid_names,out_col]=pipe.predict(Data_Prediksi.loc[valid_names,'teks_model'])
pred_cols=['pred_'+name for name in pipelines]
def vote(row):
    v=[x for x in row if x in LABELS]
    if not v:return ''
    return Counter(v).most_common(1)[0][0]
Data_Prediksi['pred_majority_vote']=Data_Prediksi[pred_cols].apply(vote,axis=1)
Data_Prediksi['agreement_3_model']=(Data_Prediksi[pred_cols].nunique(axis=1)==1) & valid_names
Data_Prediksi['review_priority']=np.where(~valid_names,'NAMA_KOSONG',
    np.where(~Data_Prediksi['agreement_3_model'],'PERBEDAAN_PREDIKSI',
    np.where(Data_Prediksi['label_verified'].isin(LABELS),'SUDAH_TERVERIFIKASI',
    np.where(Data_Prediksi['label_auto'].eq(''),'BELUM_ADA_LABEL','LABEL_AUTO_BELUM_DIVERIFIKASI'))))
# Output lengkap menjaga kolom original dan kolom turunannya
save_csv(Data_Prediksi,'Prediksi_Seluruh_12390_Record_3_Model.csv','deployment')
filter_cols=[c for c in ['id','name','manufacturer_name','software_category_name','label_auto',
'label_verified','sumber_label_effective',*pred_cols,'pred_majority_vote','agreement_3_model','review_priority'] if c in Data_Prediksi]
save_csv(Data_Prediksi[filter_cols],'Daftar_Rekomendasi_Pemfilteran_IT.csv','deployment')
review_target=Data_Prediksi.loc[Data_Prediksi.review_priority.isin(['PERBEDAAN_PREDIKSI','BELUM_ADA_LABEL'])]
save_csv(review_target[filter_cols],'Prioritas_Tinjauan_IT.csv','deployment')
print('Seluruh record:',len(Data_Prediksi),'| Nama valid:',int(valid_names.sum()),'| Ketiga model sepakat:',int(Data_Prediksi.agreement_3_model.sum()))
display(Data_Prediksi['review_priority'].value_counts().rename_axis('prioritas').reset_index(name='jumlah'))

### Coding 21 — Ringkasan prediksi, grafik dan workbook Excel untuk tim IT

In [ ]:
summary_pred=(Data_Prediksi.pred_majority_vote.replace('','TIDAK_DAPAT_DIPREDIKSI')
              .value_counts().rename_axis('hasil_prediksi').reset_index(name='jumlah'))
save_csv(summary_pred,'Tabel_4_17_Distribusi_Prediksi_Ensemble.csv','deployment')
fig,ax=plt.subplots(figsize=(9,4.5))
ax.bar(summary_pred.hasil_prediksi,summary_pred.jumlah)
ax.set(title='Gambar 4.14 — Distribusi Prediksi Mayoritas 3 Model',ylabel='Jumlah record')
ax.tick_params(axis='x',rotation=10);savefig('Gambar_4_14_Prediksi_Mayoritas.png','deployment',fig)

# Sheet terpisah: cukup untuk keperluan filter dan review, tanpa menjejalkan kolom raw di panel utama.
excel_path=FOLDERS['deployment']/'Hasil_Rekomendasi_3_Model.xlsx'
with pd.ExcelWriter(excel_path,engine='openpyxl') as wr:
    Data_Prediksi[filter_cols].to_excel(wr,sheet_name='Rekomendasi_IT',index=False)
    summary_pred.to_excel(wr,sheet_name='Distribusi_Prediksi',index=False)
    metrics.to_excel(wr,sheet_name='Evaluasi_3_Model',index=False)
    label_dist.to_excel(wr,sheet_name='Label_Awal',index=False)
    review_target[filter_cols].head(5000).to_excel(wr,sheet_name='Prioritas_Review',index=False)
    for ws in wr.book:
        ws.freeze_panes='B2';ws.auto_filter.ref=ws.dimensions
        for cell in ws[1]:cell.fill=PatternFill('solid',fgColor='183B5B');cell.font=Font(color='FFFFFF',bold=True)
        for c in ws.columns:
            letter=get_column_letter(c[0].column)
            ws.column_dimensions[letter].width=min(54,max(14,max(len(str(x.value or '')) for x in list(c)[:100])+3))
print('Excel:',excel_path)

### Coding 22 — File narasi BAB IV dan dokumentasi batasan penelitian

Narasi yang dibuat bersifat **draft berbasis angka aktual**; kalimat otomatis tetap perlu disesuaikan dengan BAB I–III dan keputusan reviewer. Narasi tidak akan menyebut label heuristik sebagai ground truth.

In [ ]:
counts=Data_Proses.label_auto.value_counts()
mode_note=(
 'Evaluasi saat ini dilakukan terhadap label otomatis berbasis aturan sementara dan hanya menunjukkan kemampuan model meniru aturan. **Angka metrik BELUM layak disebut akurasi terhadap ground truth**. Verifikasi label data uji oleh pihak IT diperlukan sebelum hasil evaluasi utama dipakai dalam skripsi.'
 if MODE_EVALUASI.startswith('DEMO') else
 'Evaluasi dilakukan pada sampel uji berlabel yang telah diverifikasi secara independen. Sebagian data training dapat berasal dari label awal heuristik; rincian sumber label dan jumlah verifikasi wajib dijelaskan.'
)
metric_table=metrics[['model','accuracy','precision_macro','recall_macro','f1_macro']].copy()
for c in ['accuracy','precision_macro','recall_macro','f1_macro']:
    metric_table[c]=metric_table[c].map(lambda z:f'{z:.4f}')
md_report=f"""# Draf BAB IV — Software Inventory GLPI

**Status evaluasi:** `{MODE_EVALUASI}`  
**Asal data:** `{input_path.name}` (SHA256: `{source_hash}`)

## 4.1 Business Understanding
Penelitian bertujuan membantu proses pemilahan entri perangkat lunak pada GLPI menjadi kategori INCLUDE dan EXCLUDE_CANDIDATE, sebagai rekomendasi untuk peninjauan IT. Gambar 4.1 memperlihatkan alur enam tahap CRISP-DM.

## 4.2 Data Understanding
Dataset awal terdiri atas **{len(Data_Mentah):,} entri** dan **{Data_Mentah.shape[1]} atribut**. Pemeriksaan awal menemukan **{int((~non_empty).sum())} nama kosong**, **{int(raw_name.duplicated().sum())} nama berulang secara eksak** (jumlah record setelah kemunculan pertama), dan **{Data_Mentah['id'].nunique():,} ID berbeda**. Pemeriksaan nilai kosong disajikan pada Gambar 4.2; distribusi kategori dan produsen masing-masing pada Gambar 4.3 dan Gambar 4.4. **Jumlah nama sama tidak berarti data boleh langsung dihapus**, karena ID GLPI masih berbeda.

## 4.3 Data Preparation
Preprocessing yang diterapkan adalah Unicode normalization, case folding, pembersihan karakter whitespace dan spasi ganda, tanpa mengubah `name` asli atau menghilangkan angka versi. Ditemukan **{int(Data_Proses.nama_berubah.sum()):,} nama berubah bentuk penulisan**. Dataset setelah cleaning memiliki **{Data_Proses.name_clean.nunique():,} nama berbeda**, termasuk entri kosong. Hasil contoh terdapat di Tabel 4.8 dan grafik dampak preprocessing di Gambar 4.7.

Label awal dari heuristik kata kunci menghasilkan **{counts.get('INCLUDE',0):,} record INCLUDE**, **{counts.get('EXCLUDE_CANDIDATE',0):,} record EXCLUDE_CANDIDATE**, dan **{counts.get('',0):,} record belum berlabel/REVIEW**. Label ini adalah **usulan eksperimen, bukan keputusan tim IT**; template validasi ahli disediakan di folder 03_Data_Preparation. **Jumlah label ahli yang benar-benar diterima pada saat menjalankan notebook: {verified_count:,} record.**

## 4.4 Modeling
Unit pemodelan dibentuk per nama hasil cleaning untuk mengurangi kebocoran duplikasi nama antara train dan test. Jumlah unit siap model adalah **{len(unit_model):,}** dengan **{len(konflik_df):,} kelompok nama bermasalah akibat konflik label**. Pembagian menghasilkan **{len(train_units):,} nama unik training** dan **{len(test_units):,} nama unik testing**, rasio mendekati 80:20 menurut mode evaluasi. Ketiga pipeline menggunakan TF-IDF dengan fitur unigram–bigram dan model Logistic Regression, Linear SVM, dan Multinomial Naive Bayes.

## 4.5 Evaluation
{mode_note}

Perbandingan metrik yang dihasilkan dari kode:

{metric_table.to_markdown(index=False)}

Confusion matrix untuk setiap model tersimpan sebagai Gambar 4.11; perbandingan metrik ditampilkan pada Gambar 4.12.

## 4.6 Deployment
Ketiga model tersimpan dalam format `.joblib` dan digunakan untuk memprediksi dataset ekspor GLPI, **tanpa mengubah isi sistem GLPI**. Sebanyak **{int(valid_names.sum()):,} record bernama tidak kosong** bisa diprediksi; **{int(Data_Prediksi.agreement_3_model.sum()):,} record** memperoleh prediksi yang sama dari ketiga model. Perbedaan prediksi diberi prioritas tinjauan oleh tim IT. Rekap distribusi prediksi terdapat pada Gambar 4.14.

## Batasan yang harus tetap ditulis
- Label heuristik bukan ground truth. Mode DEMO hanya menunjukkan kemampuan mempelajari pola usulan label, bukan ketepatan klasifikasi nyata.
- Penerimaan dan koreksi hasil perlu diputuskan oleh ahli IT; jangan menulis bahwa telah diverifikasi bila belum dilakukan.
- Label EXCLUDE_CANDIDATE berarti kandidat tidak ditampilkan dalam daftar utama, **bukan** otomatis dihapus dari GLPI.
- Unit evaluasi adalah nama software unik setelah cleaning, sementara data ekspor awal tetap utuh sebanyak {len(Data_Mentah):,} baris.
- Tidak ada pelatihan menggunakan file labeling 2.000 sebelumnya; semua label otomatis percobaan ini dibentuk dari Data_Mentah.
"""
report_path=FOLDERS['bab4']/'Draft_Narasi_BAB_IV.md'
report_path.write_text(md_report,encoding='utf-8')
readme=f"""PANDUAN FOLDER HASIL GLPI — CRISP-DM
Status evaluasi : {MODE_EVALUASI}
Input data      : {input_path.name}
Jumlah record   : {len(Data_Mentah)}

00_Dokumentasi         : metadata SHA256 sumber dan README metodologi
01_Business_Understanding: tujuan dan alur CRISP-DM (PNG)
02_Data_Understanding  : ringkasan statistik, data kosong, duplikat, kategori, manufacturer
03_Data_Preparation    : data clean, label otomatis awal, template verifikasi Pak Widi
04_Modeling            : train/test, TF-IDF, model tersimpan, validasi silang
05_Evaluation          : metrik, classification report, confusion matrices, fitur penting
06_Deployment          : prediksi seluruh baris, Excel rekomendasi dan daftar tinjauan
07_Ringkasan_BAB_IV    : draf narasi berdasarkan hasil eksekusi notebook

PENTING:
1. Tanpa label expert, hasil metrik hanya DEMO / PROXY terhadap aturan otomatis.
2. Untuk verifikasi, edit template dan simpan sebagai Label_Terverifikasi.xlsx di folder kerja; rerun seluruh notebook.
3. Jangan mengklaim validasi ahli bila belum dilakukan.
4. Raw input tidak ditimpa, semua berkas baru masuk Hasil.
5. Grafik bersifat siap insert, narasi dan interpretasinya tetap perlu diperiksa.
"""
(FOLDERS['dokumentasi']/'README_PENTING.txt').write_text(readme,encoding='utf-8')
(FOLDERS['dokumentasi']/'konfigurasi_eksperimen.json').write_text(json.dumps({
 'seed':SEED,'mode_evaluasi':MODE_EVALUASI,'label_source':'regex_examples_not_original_GLPI_67_rules',
 'model_names':list(pipelines),'tfidf':{k:str(v) for k,v in TFIDF_PARAMS.items()},
 'raw_rows':len(Data_Mentah),'verified_records':verified_count,
 'train_unique_names':len(train_units),'test_unique_names':len(test_units)},indent=2),encoding='utf-8')
print('Narasi BAB IV:',report_path)

### Coding 23 — Mengarsipkan semua output menjadi satu file ZIP

Setelah dieksekusi di Colab, buka panel **Files**, temukan folder `Hasil` dan ZIP `Hasil_GLPI_CRISP_DM.zip`. Bila tidak muncul, aktifkan ulang panel Files. File mentah tetap tidak diganti.

In [ ]:
zip_path=Path('Hasil_GLPI_CRISP_DM.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(DIR.rglob('*')):
        if p.is_file():z.write(p,arcname=str(p))
print('SELESAI: Seluruh hasil berada dalam',DIR.resolve())
print('ZIP:',zip_path.resolve(),'| total berkas:',sum(p.is_file() for p in DIR.rglob('*')))
print('MODE EVALUASI:',MODE_EVALUASI)
# Opsional di Google Colab: hapus tanda # pada baris berikut untuk mengunduh ZIP otomatis
# from google.colab import files; files.download(str(zip_path))

---
### Checklist untuk menyusun BAB IV
- [ ] Jalankan semua sel hingga Coding 23.
- [ ] Masukkan grafik Gambar 4.1–4.14 sesuai subbab; PNG sudah tersedia di `Hasil`.
- [ ] Periksa ringkasan `Hasil/07_Ringkasan_BAB_IV/Draft_Narasi_BAB_IV.md`.
- [ ] Jika memakai hasil final, **minta Pak Widi memverifikasi label**, impor file yang sudah diisi, lalu jalankan ulang.
- [ ] Pastikan tabel hasil evaluasi memakai label acuan yang benar-benar telah diverifikasi; jangan memakai skor DEMO untuk klaim accuracy final.
- [ ] Ketika membahas deployment, sebut **rekomendasi** (bukan filter/penonaktifan otomatis GLPI).